<!-- TARGET: public course-materials repo, sessions/m07/lab.ipynb. Copy to work/m07/ before editing. -->
# Module 7 · Lab: two charts to fix, one to check

**Python for Data Science · ESADE MIBA · Wed 25 Nov 2026**

Copy this notebook to `work/m07/lab.ipynb` and work on the copy. About 35 minutes; finish at home if needed.

Two charts for the course project arrive from a colleague. Each one shows correct numbers and
misleads. Fix both, then prove that one of them shows what it says. Every chart needs a **title
that says the finding** and **axis labels in words**.

Fill in each cell with `...`, then run the **check** cell below it. Silence means correct.

Every chart starts from the **order mart** (`data/mart/orders_enriched.parquet`) and, for maps,
the state outlines in `data/geo/brazil-states.geojson`. If the cell below stops with an
`AssertionError`, copy `sessions/m05/lab-solution.ipynb` to `work/m05/`, Run All, and come back.
The cell also sets the ESADE chart style from M06 and keeps `delivered`: delivered orders with a
delivery date.

In [ ]:
import json
from pathlib import Path

import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
import plotly.io as pio

RAW = Path("../../data/raw")
MART_FILE = Path("../../data/mart/orders_enriched.parquet")
GEO_FILE = Path("../../data/geo/brazil-states.geojson")

assert MART_FILE.exists(), (
    "No order mart yet. Copy sessions/m05/lab-solution.ipynb to work/m05/, Run All, then come back."
)
assert GEO_FILE.exists(), "No map file. Pull the course repository: it ships data/geo/brazil-states.geojson."
mart = pd.read_parquet(MART_FILE)
assert mart.shape == (99441, 13), (
    f"Your mart has shape {mart.shape}, expected (99441, 13). Run sessions/m05/lab-solution.ipynb to rebuild it."
)
with open(GEO_FILE) as geo_file:
    brazil_states = json.load(geo_file)

ESADE_BLUE = "#00205B"
ESADE_LIGHT_BLUE = "#4A9EFF"
GREY = "#B8BEC8"
pio.templates["esade"] = go.layout.Template(
    layout=dict(
        font=dict(family="Arial", color="#2B2B2B"),
        title=dict(font=dict(color=ESADE_BLUE, size=20)),
        colorway=[ESADE_BLUE, ESADE_LIGHT_BLUE, "#E07A1F", "#7A869A", "#2CA58D", "#C1272D"],
    )
)
pio.templates.default = "plotly_white+esade"

delivered = mart[(mart["order_status"] == "delivered") & mart["is_late"].notna()]

delivered = delivered.assign(delivery=delivered["is_late"].map({True: "late", False: "on time"}))


def check_labelled(fig, *column_names):
    assert fig.layout.title.text, "Give the chart a title that says the finding."
    axis_titles = [fig.layout.xaxis.title.text, fig.layout.yaxis.title.text]
    for name in column_names:
        assert name not in axis_titles, f"Axis still shows the column name {name!r}: use labels=."

## Q1 · The score gap (10 min)

The colleague's chart: mean review score, late vs on time. Run it.

In [ ]:
score_gap = (
    delivered.groupby("delivery")
    .agg(orders=("order_id", "size"), mean_score=("review_score", "mean"))
    .reset_index()
)
fig_bad_gap = px.bar(score_gap, x="delivery", y="mean_score", title="Late orders are a disaster")
fig_bad_gap.update_layout(yaxis_range=[2.2, 4.4])
fig_bad_gap

In one sentence: what is wrong with it?

*Your answer:* ...

Fix it in `fig_gap`: a bar chart of the same table whose bars start at zero, with a title that
says the finding and axis labels in words.

In [ ]:
fig_gap = ...
fig_gap

In [ ]:
assert fig_gap.data[0].type == "bar"
y_range = fig_gap.layout.yaxis.range
assert y_range is None or y_range[0] == 0, "Bars start at zero."
plotted = dict(zip(fig_gap.data[0].x, fig_gap.data[0].y))
assert round(plotted["late"], 2) == 2.27 and round(plotted["on time"], 2) == 4.29
assert "disaster" not in fig_gap.layout.title.text.lower(), "Say the finding, with its size."
check_labelled(fig_gap, "delivery", "mean_score")

## Q2 · Two measures, two axes (15 min)

The colleague's second chart: delivered orders per month (bars, left axis) and their mean review
score (line, right axis). It is built with a second plotly interface that we don't use; just run
it.

In [ ]:
from plotly.subplots import make_subplots

monthly = (
    delivered[delivered["purchase_month"].between("2017-01", "2018-08")]
    .groupby("purchase_month")
    .agg(orders=("order_id", "size"), mean_score=("review_score", "mean"))
    .reset_index()
)
fig_dual = make_subplots(specs=[[{"secondary_y": True}]])
fig_dual.add_bar(x=monthly["purchase_month"], y=monthly["orders"], name="orders")
fig_dual.add_scatter(x=monthly["purchase_month"], y=monthly["mean_score"], name="mean score", secondary_y=True)
fig_dual.update_yaxes(range=[3.7, 4.3], secondary_y=True)
fig_dual.update_layout(title="As orders grow, satisfaction falls")
fig_dual

In one sentence: why can't the reader trust the relationship this chart suggests?

*Your answer:* ...

Redraw it in `fig_panels`: **two panels** that share the x axis (`facet_row`), one measure per
panel, each panel with its own y scale. Hint: `melt` `monthly` into long format first (call it
`monthly_long`, with columns `purchase_month`, `measure`, `value`). Map `measure` to `color`
too: each line then carries its measure's name.

In [ ]:
monthly_long = ...
fig_panels = ...
fig_panels

In [ ]:
assert len(monthly_long) == 2 * len(monthly) == 40, "One row per month × measure."
assert {trace.name for trace in fig_panels.data} == {"orders", "mean_score"}, "One line per measure, coloured by measure."
assert {trace.yaxis for trace in fig_panels.data} == {"y", "y2"}, "One panel per measure (facet_row)."
assert fig_panels.layout.yaxis.matches is None and fig_panels.layout.yaxis2.matches is None, (
    "Each panel needs its own y scale: fig.update_yaxes(matches=None)."
)
assert fig_panels.layout.title.text

## Q3 · Check the panels, and share them (10 min)

Prove that `fig_panels` shows the right numbers. For **each** trace, build a Series of its values
indexed by month, and `assert` it matches a fresh `groupby` of `delivered`. Then save the chart as
`score-panels.html` with `write_html` and `assert` the file exists.

Hint: `fig_panels.data` holds the traces. For each `trace`, `trace.name` is the measure,
`trace.x` the months and `trace.y` the values.

In [ ]:
fresh = ...
for trace in fig_panels.data:
    ...

html_file = ...

In [ ]:
assert Path("score-panels.html").exists(), "Write the chart with fig_panels.write_html(...)."
assert fresh["orders"].sum() == delivered["purchase_month"].between("2017-01", "2018-08").sum()

## For the project

Your M06 charts plus today's fixes are the evidence for M08 this afternoon. In two sentences:
which chart would you put first in front of the operations director, and why?

*Your answer:* ...